# Liputan6: factuality-ranked train → residual SBERT-GCN
Train on the **10,000 highest `prob_yes_factuality` scores among AMR-complete train documents**.
Use canonical **dev for validation**, and **test only for final evaluation**.

- Existing 768-dimensional features are assumed to include position information; provenance is unverified. Do not append more positional dimensions.
- Existing NPY/NPZ adjacency weights are used unchanged. PyG handles self-loop addition and GCN normalization internally.
- Generate greedy **ROUGE-1 F1** binary oracle labels from source sentences and `clean_summary`, with at most two selected oracle sentences (stop earlier if no gain).
- Model: GCN 768→256→256, projected outer residual, dropout, MLP 256→128→1.
- Prediction: ranked top-2 with trigram blocking (no positive-score threshold), restore source order.
- Optimize weighted BCE; choose the best checkpoint by **dev macro ROUGE-1 F1**, not sentence-selection F1.

This notebook is self-contained. Use the local `generate_amr` kernel. No package installation or model download is performed here.

Training candidates must have every canonical sentence graph in train.sqlite. Incomplete/misaligned documents are excluded before taking the top 10,000. This is structural completeness, not an AMR semantic-quality filter.

Current experiment E002 caps greedy labels at two sentences. E001 unlimited-label results and source snapshot are preserved separately; see EXPERIMENTS.md.


In [ ]:
import hashlib
import json
import math
import os
import random
import time
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv
from rouge_score import rouge_scorer
from tqdm.auto import tqdm

ROUGE = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)
ORACLE_ROUGE = rouge_scorer.RougeScorer(["rouge1"], use_stemmer=False)

import sqlite3
from contextlib import closing, contextmanager


In [ ]:
# Edit paths here. Train shards are read sequentially; matrices prefer {split}.npz.
ROOT = Path(r"D:\Github\generate_amr")
DATA_ROOT = ROOT / "data/liputan6"
FEATURE_PATHS = {
    "train": [
        DATA_ROOT / "features/drive-download-20261010T052944Z-1-001/train_1.pt",
        DATA_ROOT / "features/drive-download-20261010T052944Z-1-003/train_2.pt",
        DATA_ROOT / "features/drive-download-20261010T052944Z-1-002/train_3.pt",
        DATA_ROOT / "features/drive-download-20261010T052944Z-1-004/train_4.pt",
    ],
    "dev": DATA_ROOT / "features/dev.pt",
    "test": DATA_ROOT / "features/test.pt",
}
SOURCE_ROOT = DATA_ROOT / "source/Liputan6/liputan6/liputan6_data/canonical"
ADJACENCY_ROOT = DATA_ROOT / "adjacency_matrices"
TRAIN_AMR_DB = DATA_ROOT / "amr_graphs/sqlite/train.sqlite"
FACTUALITY_CSV = DATA_ROOT / "faktualitas_lip6_v2.csv"
FACTUALITY_ID_COLUMN = "id"
FACTUALITY_SPLIT_COLUMN = "split"
FACTUALITY_SCORE_COLUMN = "prob_yes_factuality"  # NOT the binary factuality column.
TRAIN_DOCUMENTS = 10_000
OUTPUT = ROOT / "runs/pyg-gcn-top10000-factuality-oracle2"

INPUT_DIM = 768
HIDDEN_DIM = 256
DROPOUT = 0.3
EPOCHS = 10
BATCH_SIZE = 16
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.0
GRADIENT_CLIP = 1.0
POS_WEIGHT = None  # None: compute negative/positive label ratio on TRAIN only.
PATIENCE = 0       # 0 disables early stopping.
TOP_K = 2
TRIGRAM_BLOCKING = True
BASELINE = False   # True removes inter-sentence edges but keeps the architecture.
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RUN_TEST_EVALUATION = False  # Set True only after model/selection choices are fixed.
ORACLE_EPSILON = 1e-12
LOG_EVERY = 100
ORACLE_MAX_SENTENCES = 2  # None reproduces unlimited labeling; use a new OUTPUT per experiment.
ORACLE_SETTINGS = {
    "algorithm": "greedy-best-improvement", "metric": "rouge1_f1",
    "max_sentences": ORACLE_MAX_SENTENCES, "use_stemmer": False,
    "sentence_order": "source", "tie_break": "lowest-sentence-index",
    "epsilon": ORACLE_EPSILON, "rouge_score_version": version("rouge-score"),
}
print("Device:", DEVICE, "| train documents:", TRAIN_DOCUMENTS)


## Data contracts and reproducible selection

The factuality CSV contains `id, split, factuality, prob_no_factuality, prob_yes_factuality`.
Only rows with `split == train` are ranked. Dev/test factuality scores do not filter evaluation.

Features may be:
- a trusted `.pt` list of `{"id": document_id, "embeddings": tensor[n,768]}`; or
- a directory of trusted `{doc_id}.pt` files, each containing a tensor.

A monolithic train `.pt` is fully deserialized before retaining selected records; allow sufficient RAM.
Source JSON supplies `clean_article` and `clean_summary`. Existing `extractive_summary` is not used as training supervision here.

Candidates are ranked by factuality and checked for exact AMR coverage before selection. Incomplete/misaligned documents are excluded. Missing features or matrices for selected complete documents still cause failure, rather than silently changing the cohort.
Dev/test coverage follows all canonical JSON files in the corresponding split, not only whichever feature records happen to exist.
Row counts are checked, but features lack sentence IDs: correct row ordering remains an upstream assumption.

Adjacency uses split.npz when present, otherwise split/doc_id.npy. Train feature inputs may be a list of record-list shards, loaded one at a time. Selected tensors remain in RAM.

Oracle labels are capped by ORACLE_MAX_SENTENCES (currently 2). This does not force two positives. Automatic class weights are recomputed from the capped train labels; dev sentence micro-F1 refers to these capped labels.


In [ ]:
def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    try:
        temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def safe_id(value):
    value = str(value)
    if not value or value in {".", ".."} or any(c in value for c in ("/", "\\", "\0")):
        raise ValueError(f"Unsafe/empty document ID: {value!r}")
    return value


def canonical_ids(split):
    folder = SOURCE_ROOT / split
    if not folder.is_dir():
        raise FileNotFoundError(f"Canonical source directory missing: {folder}")
    ids = sorted(path.stem for path in folder.glob("*.json"))
    if not ids:
        raise ValueError(f"No canonical JSON files in {folder}")
    return ids


def train_amr_statistics():
    """Exact index coverage follows from the schema's unique (split,doc_id,sent_idx)."""
    path = Path(TRAIN_AMR_DB)
    if not path.is_file():
        raise FileNotFoundError(f"Train AMR SQLite missing: {path}")
    with closing(sqlite3.connect(path.resolve().as_uri() + "?mode=ro", uri=True)) as db:
        version_row = db.execute(
            "SELECT value FROM storage_metadata WHERE key='schema_version'").fetchone()
        if version_row != ("amr-documents-v1",):
            raise ValueError(f"Unsupported AMR schema: {version_row}")
        rows = db.execute("""
            SELECT doc_id, COUNT(*), COUNT(DISTINCT sent_idx), MIN(sent_idx), MAX(sent_idx),
                   SUM(CASE WHEN sentence_id != doc_id || '_' || sent_idx
                            OR length(trim(amr_penman)) = 0 THEN 1 ELSE 0 END)
            FROM graphs WHERE split='train' GROUP BY doc_id
        """).fetchall()
    return {str(row[0]): row[1:] for row in rows}


def select_train_manifest():
    columns = [FACTUALITY_ID_COLUMN, FACTUALITY_SPLIT_COLUMN, FACTUALITY_SCORE_COLUMN]
    frame = pd.read_csv(FACTUALITY_CSV, dtype={
        FACTUALITY_ID_COLUMN: "string", FACTUALITY_SPLIT_COLUMN: "string"})
    if any(column not in frame.columns for column in columns):
        raise ValueError(f"Required factuality columns: {columns}; found {list(frame.columns)}")
    frame = frame.loc[frame[FACTUALITY_SPLIT_COLUMN] == "train", columns].copy()
    if frame[columns].isna().any().any():
        raise ValueError("Missing ID, split, or factuality score in TRAIN rows")
    frame[FACTUALITY_ID_COLUMN] = frame[FACTUALITY_ID_COLUMN].map(safe_id)
    if frame[FACTUALITY_ID_COLUMN].duplicated().any():
        raise ValueError("Duplicate document IDs in TRAIN factuality rows")
    scores = pd.to_numeric(frame[FACTUALITY_SCORE_COLUMN], errors="raise")
    if not np.isfinite(scores).all() or not scores.between(0, 1).all():
        raise ValueError("Factuality probabilities must be finite values in [0,1]")
    frame[FACTUALITY_SCORE_COLUMN] = scores
    if not isinstance(TRAIN_DOCUMENTS, int) or TRAIN_DOCUMENTS < 1:
        raise ValueError("TRAIN_DOCUMENTS must be a positive integer")
    if len(frame) < TRAIN_DOCUMENTS:
        raise ValueError(f"Need {TRAIN_DOCUMENTS} scored train documents; found {len(frame)}")
    ranked = frame.sort_values(
        [FACTUALITY_SCORE_COLUMN, FACTUALITY_ID_COLUMN],
        ascending=[False, True], kind="mergesort")
    print("Checking train AMR index coverage...", flush=True)
    statistics = train_amr_statistics()
    selected, checked = [], []
    started = time.monotonic()
    for rank, record in enumerate(ranked.to_dict("records"), 1):
        doc_id = record[FACTUALITY_ID_COLUMN]
        stats = statistics.get(doc_id)
        expected, reason = None, "no_amr_graphs"
        if stats is not None:
            source_path = SOURCE_ROOT / "train" / f"{doc_id}.json"
            if not source_path.is_file():
                reason = "missing_canonical_source"
            else:
                source = json.loads(source_path.read_text(encoding="utf-8-sig"))
                article = source.get("clean_article")
                if not isinstance(article, list) or not article:
                    raise ValueError(f"Invalid clean_article: {source_path}")
                expected = len(article)
                count, distinct, minimum, maximum, bad_rows = stats
                reason = ("complete" if count == distinct == expected
                          and minimum == 0 and maximum == expected - 1 and bad_rows == 0
                          else "incomplete_or_misaligned_amr")
        audit = dict(record, factuality_rank=rank, expected_sentences=expected,
                     amr_graphs=stats[0] if stats else 0, reason=reason)
        checked.append(audit)
        if reason == "complete":
            selected.append(audit)
        if rank == 1 or rank % 1000 == 0:
            print(f"ELIGIBILITY: checked={rank:,} | complete selected={len(selected):,}"
                  f"/{TRAIN_DOCUMENTS:,} | elapsed={time.monotonic()-started:.0f}s", flush=True)
        if len(selected) == TRAIN_DOCUMENTS:
            break
    OUTPUT.mkdir(parents=True, exist_ok=True)
    audit_frame = pd.DataFrame(checked)
    audit_frame.to_csv(OUTPUT / "train_amr_eligibility_checked.csv", index=False)
    audit_frame.loc[audit_frame["reason"] != "complete"].to_csv(
        OUTPUT / "train_amr_excluded.csv", index=False)
    print(f"AMR eligibility: checked {len(checked):,}, selected {len(selected):,}, "
          f"excluded {len(checked)-len(selected):,}", flush=True)
    if len(selected) < TRAIN_DOCUMENTS:
        raise ValueError(f"Need {TRAIN_DOCUMENTS} complete train documents; found {len(selected)}")
    manifest = pd.DataFrame(selected)
    manifest.insert(0, "rank", range(1, len(manifest) + 1))
    return manifest


def feature_input_paths(spec):
    return [Path(path) for path in spec] if isinstance(spec, (list, tuple)) else [Path(spec)]


def feature_records(spec, ids):
    for path in feature_input_paths(spec):
        print(f"Reading features: {path}", flush=True)
        if path.is_dir():
            for doc_id in ids:
                yield {"id": doc_id, "embeddings": torch.load(
                    path / f"{doc_id}.pt", map_location="cpu", weights_only=True)}
        else:
            records = torch.load(path, map_location="cpu", weights_only=True)
            if not isinstance(records, list):
                raise ValueError(f"{path}: expected a list of id/embeddings records")
            yield from records
            del records


@contextmanager
def adjacency_source(split):
    archive_path = ADJACENCY_ROOT / f"{split}.npz"
    if archive_path.is_file():
        with np.load(archive_path, allow_pickle=False) as archive:
            if len(archive.files) != len(set(archive.files)):
                raise ValueError(f"Duplicate document keys: {archive_path}")
            yield archive
    else:
        yield ADJACENCY_ROOT / split


def read_adjacency(source, doc_id):
    if isinstance(source, Path):
        return np.load(source / f"{doc_id}.npy", allow_pickle=False)
    return source[doc_id]


def load_feature_map(path, ids):
    wanted = set(ids)
    result = {}
    records = feature_records(path, ids)
    for record in records:
        if not isinstance(record, dict) or not {"id", "embeddings"} <= record.keys():
            raise ValueError(f"{path}: invalid feature record")
        doc_id = safe_id(record["id"])
        if doc_id not in wanted:
            continue
        if doc_id in result:
            raise ValueError(f"{path}: duplicate feature ID {doc_id}")
        x = record["embeddings"]
        if not isinstance(x, torch.Tensor) or x.ndim != 2 or x.shape[1] != INPUT_DIM:
            raise ValueError(f"{doc_id}: expected tensor[n,{INPUT_DIM}]")
        if not torch.isfinite(x).all():
            raise ValueError(f"{doc_id}: nonfinite features")
        result[doc_id] = x.float().contiguous()
    missing = wanted - result.keys()
    if missing:
        raise ValueError(f"{path}: missing {len(missing)} required feature IDs; examples {sorted(missing)[:10]}")
    return result


def source_texts(source):
    def sentence(tokens):
        if not isinstance(tokens, list) or not tokens or not all(isinstance(t, str) for t in tokens):
            raise ValueError("Expected nonempty token lists in canonical source")
        text = " ".join(tokens)
        if not text.strip():
            raise ValueError("Empty sentence")
        return text
    if not isinstance(source["clean_article"], list) or not source["clean_article"]:
        raise ValueError("Empty/invalid clean_article")
    if not isinstance(source["clean_summary"], list) or not source["clean_summary"]:
        raise ValueError("Empty/invalid clean_summary")
    sentences = [sentence(tokens) for tokens in source["clean_article"]]
    reference = " ".join(sentence(tokens) for tokens in source["clean_summary"])
    return sentences, reference


def greedy_oracle(sentences, reference):
    selected, best_score = [], 0.0
    limit = ORACLE_SETTINGS["max_sentences"]
    if limit is not None and (type(limit) is not int or limit < 1):
        raise ValueError("Oracle sentence cap must be None or a positive integer")
    budget = len(sentences) if limit is None else min(limit, len(sentences))
    while len(selected) < budget:
        best_index, next_score = None, best_score
        for index in range(len(sentences)):
            if index in selected:
                continue
            candidate = " ".join(sentences[i] for i in sorted(selected + [index]))
            score = ORACLE_ROUGE.score(reference, candidate)["rouge1"].fmeasure
            if score > next_score + ORACLE_EPSILON:
                best_index, next_score = index, score
        if best_index is None:
            break
        selected.append(best_index)
        best_score = next_score
    return sorted(selected)


def cached_oracle(split, doc_id, sentences, reference):
    signature = hashlib.sha256(json.dumps(
        {"sentences": sentences, "reference": reference, "settings": ORACLE_SETTINGS},
        ensure_ascii=False, sort_keys=True).encode("utf-8")).hexdigest()
    path = OUTPUT / "oracle_cache" / split / f"{doc_id}.json"
    if path.is_file():
        cached = json.loads(path.read_text(encoding="utf-8"))
        indices = cached.get("indices")
        if (cached.get("signature") == signature and isinstance(indices, list)
                and all(type(i) is int and 0 <= i < len(sentences) for i in indices)
                and indices == sorted(set(indices))
                and (ORACLE_SETTINGS["max_sentences"] is None
                     or len(indices) <= ORACLE_SETTINGS["max_sentences"])):
            return indices
    indices = greedy_oracle(sentences, reference)
    atomic_json(path, {"signature": signature, "indices": indices, "settings": ORACLE_SETTINGS})
    return indices


def load_split(split, ids):
    print(f"Loading {split}: {len(ids):,} documents", flush=True)
    features = load_feature_map(FEATURE_PATHS[split], ids)
    examples, errors = [], []
    start, last_log = time.monotonic(), time.monotonic()
    with adjacency_source(split) as matrices:
        for number, doc_id in enumerate(tqdm(ids, desc=f"Prepare {split}", unit="doc"), 1):
            try:
                source = json.loads((SOURCE_ROOT / split / f"{doc_id}.json").read_text(encoding="utf-8-sig"))
                sentences, reference = source_texts(source)
                n = len(sentences)
                x = features.pop(doc_id)
                a = read_adjacency(matrices, doc_id)
                if x.shape[0] != n or a.shape != (n, n):
                    raise ValueError(f"Alignment mismatch: features={tuple(x.shape)}, adjacency={a.shape}, sentences={n}")
                if not np.isfinite(a).all() or not np.allclose(a, a.T, rtol=0, atol=1e-5):
                    raise ValueError("Adjacency must be finite and symmetric")
                if (a < 0).any():
                    raise ValueError("Negative adjacency weights unsupported by this GCN normalization")
                # Preserve stored weights/connectivity; do not threshold or re-normalize here.
                adjacency = torch.from_numpy(a.astype(np.float32))
                edge_index = adjacency.nonzero(as_tuple=False).t().contiguous()
                edge_weight = adjacency[edge_index[0], edge_index[1]]
                gold = cached_oracle(split, doc_id, sentences, reference)
                y = torch.zeros(n, dtype=torch.float32)
                y[gold] = 1
                graph = Data(x=x, edge_index=edge_index, edge_weight=edge_weight, y=y)
                examples.append({"id": doc_id, "graph": graph, "sentences": sentences,
                                 "reference": reference, "gold": gold})
            except (ValueError, KeyError, OSError, TypeError) as exc:
                errors.append({"id": doc_id, "error": str(exc)})
            now = time.monotonic()
            if number == 1 or number % LOG_EVERY == 0 or number == len(ids) or now - last_log >= 60:
                print(f"PREPARE {split}: {number:,}/{len(ids):,} | valid={len(examples):,} "
                      f"errors={len(errors):,} | elapsed={now-start:.0f}s", flush=True)
                last_log = now
    atomic_json(OUTPUT / f"{split}_input_errors.json", errors)
    if errors:
        raise ValueError(f"{split}: {len(errors)} invalid required documents; see {OUTPUT / (split + '_input_errors.json')}")
    return examples


## Residual GCN, scoring, and reference-free prediction
The second GCN output is added to a learned projection of the original input,
then passed through dropout and an MLP. BCE receives logits, not sigmoid outputs.
Ranking logits is equivalent to ranking sigmoid probabilities. No 0.5 filter is used.

Trigram blocking compares lowercased whitespace tokens and skips a candidate if it
shares any trigram with an already selected sentence. It may return fewer than two
sentences. Selection never accesses the reference or oracle labels.


In [ ]:
class SentenceScorer(nn.Module):
    def __init__(self, input_dim=768, hidden=256, dropout=0.3, baseline=False):
        super().__init__()
        self.baseline = baseline
        self.dropout = dropout
        self.conv1 = GCNConv(input_dim, hidden, cached=False, add_self_loops=True, normalize=True)
        self.conv2 = GCNConv(hidden, hidden, cached=False, add_self_loops=True, normalize=True)
        self.skip = nn.Linear(input_dim, hidden, bias=False)
        self.head = nn.Sequential(nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Linear(hidden // 2, 1))

    def forward(self, data):
        x, edges, weights = data.x, data.edge_index, data.edge_weight
        if self.baseline:
            edges = torch.empty((2, 0), dtype=torch.long, device=x.device)
            weights = None
        h = F.relu(self.conv1(x, edges, edge_weight=weights))
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = self.conv2(h, edges, edge_weight=weights) + self.skip(x)
        h = F.dropout(h, p=self.dropout, training=self.training)
        return self.head(h).squeeze(-1)


def select_sentences(logits, sentences, top_k=2, block_trigrams=True):
    if top_k < 1 or len(logits) != len(sentences):
        raise ValueError("Invalid selection dimensions/top_k")
    values = logits.detach().cpu().tolist()
    if not all(math.isfinite(score) for score in values):
        raise ValueError("Nonfinite prediction logits")
    ranked = sorted(range(len(sentences)), key=lambda index: (-values[index], index))
    chosen, used = [], set()
    for index in ranked:
        tokens = sentences[index].lower().split()
        trigrams = set(zip(tokens, tokens[1:], tokens[2:]))
        if block_trigrams and used.intersection(trigrams):
            continue
        chosen.append(index)
        used.update(trigrams)
        if len(chosen) == top_k:
            break
    return sorted(chosen)


@torch.no_grad()
def evaluate(model, examples, device, pos_weight, top_k=2, block_trigrams=True):
    if not examples:
        raise ValueError("No evaluation documents")
    model.eval()
    tp = predicted = actual = 0
    loss_sum = 0.0
    rouge_totals = {key: 0.0 for key in ("rouge1_f1", "rouge2_f1", "rougeL_f1")}
    predictions = []
    start, last_log = time.monotonic(), time.monotonic()
    for number, example in enumerate(examples, 1):
        data = example["graph"].clone().to(device)
        logits = model(data)
        loss_sum += F.binary_cross_entropy_with_logits(logits, data.y, pos_weight=pos_weight).item()
        chosen = select_sentences(logits, example["sentences"], top_k, block_trigrams)
        summary = " ".join(example["sentences"][i] for i in chosen)
        scores = ROUGE.score(example["reference"], summary)
        for name, score in scores.items():
            rouge_totals[name + "_f1"] += score.fmeasure
        gold = set(example["gold"])
        tp += len(set(chosen) & gold)
        predicted += len(chosen)
        actual += len(gold)
        predictions.append({"id": example["id"], "selected_sentences": chosen,
                            "gold_sentences": sorted(gold), "summary": summary})
        now = time.monotonic()
        if number == 1 or number % LOG_EVERY == 0 or number == len(examples) or now-last_log >= 60:
            print(f"EVALUATE: {number:,}/{len(examples):,} documents | elapsed={now-start:.0f}s", flush=True)
            last_log = now
    precision = tp / predicted if predicted else 0.0
    recall = tp / actual if actual else 0.0
    metrics = {"documents": len(examples), "loss": loss_sum / len(examples),
               "sentence_precision": precision, "sentence_recall": recall,
               "sentence_micro_f1": 2 * precision * recall / (precision + recall) if precision + recall else 0.0,
               **{key: value / len(examples) for key, value in rouge_totals.items()}}
    if not all(math.isfinite(value) for value in metrics.values()):
        raise ValueError("Nonfinite evaluation metrics")
    return metrics, predictions


## Eligibility, fixed manifests, and preparation

Rank train candidates by factuality; check exact AMR sentence-index coverage against canonical clean_article; take the first 10,000 complete documents. Save checked candidates and exclusions (lower-ranked candidates after the cutoff are not checked). Missing selected features/matrices still fail instead of silently changing the cohort. Dev/test membership is unchanged.


In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
# Some PyG CUDA scatter operations may remain nondeterministic; seeds are not a bitwise guarantee.
if EPOCHS < 1 or BATCH_SIZE < 1 or TOP_K < 1 or LOG_EVERY < 1:
    raise ValueError("EPOCHS, BATCH_SIZE, TOP_K, and LOG_EVERY must be positive")
if not FACTUALITY_CSV.is_file():
    raise FileNotFoundError(f"Factuality CSV missing: {FACTUALITY_CSV}")
OUTPUT.mkdir(parents=True, exist_ok=True)
manifest = select_train_manifest()
train_ids = manifest[FACTUALITY_ID_COLUMN].tolist()
dev_ids = canonical_ids("dev")
test_ids = canonical_ids("test")  # IDs only; test features/references are not evaluated here.
split_sets = [set(train_ids), set(dev_ids), set(test_ids)]
if any(split_sets[i] & split_sets[j] for i in range(3) for j in range(i + 1, 3)):
    raise ValueError("Document ID overlap between train/dev/test")
manifest.to_csv(OUTPUT / "selected_train_factuality.csv", index=False)
split_manifest = {"train": train_ids, "dev": dev_ids, "test": test_ids,
                  "selection": "top factuality among train documents with exact canonical AMR index coverage"}
atomic_json(OUTPUT / "split_manifest.json", split_manifest)
config = {
    "architecture": "residual-gcn-mlp-v2", "train_documents": TRAIN_DOCUMENTS,
    "split_manifest_sha256": hashlib.sha256(json.dumps(split_manifest, sort_keys=True).encode()).hexdigest(),
    "factuality_csv": str(FACTUALITY_CSV),
    "factuality_csv_sha256": hashlib.sha256(FACTUALITY_CSV.read_bytes()).hexdigest(),
    "score_column": FACTUALITY_SCORE_COLUMN,
    "features": {split: [str(p) for p in feature_input_paths(spec)] for split, spec in FEATURE_PATHS.items()},
    "train_amr_db": str(TRAIN_AMR_DB),
    "amr_eligibility": "count=distinct=n, min=0, max=n-1, valid sentence IDs; not semantic quality",
    "adjacency_storage": "prefer split.npz; fallback split/doc_id.npy",
    "source_root": str(SOURCE_ROOT), "adjacency_root": str(ADJACENCY_ROOT),
    "position_encoding": "assumed already included; unverified",
    "feature_sentence_order": "assumed canonical; no per-sentence IDs available",
    "model": {"input_dim": INPUT_DIM, "hidden": HIDDEN_DIM, "dropout": DROPOUT, "baseline": BASELINE},
    "oracle": ORACLE_SETTINGS, "selection": {"top_k": TOP_K, "block_trigrams": TRIGRAM_BLOCKING},
    "training": {"epochs": EPOCHS, "batch_size": BATCH_SIZE, "lr": LEARNING_RATE,
                 "weight_decay": WEIGHT_DECAY, "gradient_clip": GRADIENT_CLIP,
                 "requested_pos_weight": POS_WEIGHT, "patience": PATIENCE, "seed": SEED},
    "checkpoint_metric": "dev_macro_rouge1_f1",
    "packages": {name: version(name) for name in ("torch", "torch-geometric", "numpy", "rouge-score")},
}
atomic_json(OUTPUT / "config.json", config)
missing = []
for split, ids in (("train", train_ids), ("dev", dev_ids)):
    for path in feature_input_paths(FEATURE_PATHS[split]):
        if not path.exists():
            missing.append(f"{split} features: {path}")
    with adjacency_source(split) as matrices:
        keys = None if isinstance(matrices, Path) else set(matrices.files)
        for doc_id in ids:
            source_path = SOURCE_ROOT / split / f"{doc_id}.json"
            if not source_path.is_file():
                missing.append(f"{split} {doc_id} source: {source_path}")
            present = ((matrices / f"{doc_id}.npy").is_file()
                       if keys is None else doc_id in keys)
            if not present:
                missing.append(f"{split} {doc_id} adjacency")
atomic_json(OUTPUT / "missing_inputs.json", missing)
if missing:
    raise FileNotFoundError(f"{len(missing):,} missing inputs; first 10:\n" + "\n".join(missing[:10]))
train = load_split("train", train_ids)
validation = load_split("dev", dev_ids)
positive = sum(float(example["graph"].y.sum()) for example in train)
negative = sum(example["graph"].num_nodes for example in train) - positive
if positive == 0:
    raise ValueError("No positive oracle labels in train")
weight_value = (negative / positive if negative else 1.0) if POS_WEIGHT is None else float(POS_WEIGHT)
if not math.isfinite(weight_value) or weight_value <= 0:
    raise ValueError("POS_WEIGHT must be finite and positive")
pos_weight = torch.tensor(weight_value, dtype=torch.float32, device=DEVICE)
config["training"]["effective_pos_weight"] = weight_value
config["training"]["positive_labels"] = int(positive)
config["training"]["negative_labels"] = int(negative)
atomic_json(OUTPUT / "config.json", config)
print(f"Ready: train={len(train):,}, dev={len(validation):,}, held-out test={len(test_ids):,} "
      f"| pos_weight={weight_value:.4f}", flush=True)


## Train with weighted BCE; select best by dev ROUGE-1 F1
A graph mini-batch combines sentences from multiple disconnected document graphs.
Training loss is averaged over nodes; reported validation loss is a document mean.
Class weighting is calculated only from the selected train oracle labels.

Each run starts fresh; `best.pt` and `last.pt` are saved atomically. Oracle preparation
is resumable, but optimizer-state training resume is not implemented. Use a new
OUTPUT directory to retain a previous experiment. Test data never selects a checkpoint.


In [ ]:
def atomic_checkpoint(path, value):
    temporary = path.with_suffix(".pt.tmp")
    try:
        torch.save(value, temporary)
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


model = SentenceScorer(**config["model"]).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
generator = torch.Generator().manual_seed(SEED)
loader = DataLoader([example["graph"] for example in train], batch_size=BATCH_SIZE,
                    shuffle=True, generator=generator, num_workers=0)
best_rouge, bad_epochs, history = -1.0, 0, []
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = total_nodes = 0
    last_log = time.monotonic()
    for batch_number, batch in enumerate(tqdm(loader, desc=f"Epoch {epoch}/{EPOCHS}", unit="batch"), 1):
        batch = batch.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(batch)
        loss = F.binary_cross_entropy_with_logits(logits, batch.y, pos_weight=pos_weight)
        if not torch.isfinite(loss):
            raise ValueError("Nonfinite training loss")
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP, error_if_nonfinite=True)
        optimizer.step()
        total_loss += loss.item() * batch.num_nodes
        total_nodes += batch.num_nodes
        now = time.monotonic()
        if batch_number == 1 or batch_number % LOG_EVERY == 0 or batch_number == len(loader) or now-last_log >= 60:
            print(f"TRAIN epoch={epoch}: {batch_number}/{len(loader)} batches "
                  f"| loss={total_loss/total_nodes:.5f}", flush=True)
            last_log = now
    metrics, _ = evaluate(model, validation, DEVICE, pos_weight, TOP_K, TRIGRAM_BLOCKING)
    history.append({"epoch": epoch, "train_loss": total_loss / total_nodes, **metrics})
    improved = metrics["rouge1_f1"] > best_rouge + 1e-12
    if improved:
        best_rouge, bad_epochs = metrics["rouge1_f1"], 0
    else:
        bad_epochs += 1
    checkpoint = {"state_dict": model.state_dict(), "architecture": config["architecture"],
                  "config": config, "epoch": epoch, "dev_metrics": metrics}
    atomic_checkpoint(OUTPUT / "last.pt", checkpoint)
    if improved:
        atomic_checkpoint(OUTPUT / "best.pt", checkpoint)
    atomic_json(OUTPUT / "history.json", history)
    print(json.dumps(history[-1]), flush=True)
    if PATIENCE > 0 and bad_epochs >= PATIENCE:
        print(f"Early stopping after {epoch} epochs", flush=True)
        break
print("Best dev ROUGE-1 F1:", best_rouge, "| checkpoint:", OUTPUT / "best.pt", flush=True)


## Final test evaluation
Enable `RUN_TEST_EVALUATION` only after fixing all model and selection choices.
The saved best checkpoint is loaded; no optimizer update occurs. Prediction uses
only source features/graph/sentence text. The reference is used afterward for ROUGE,
and test oracle labels only for the supplementary sentence-selection F1.

ROUGE values are macro-average F1 in [0,1]; multiply by 100 for percentage reporting.
For evaluation-only use, run imports/configuration/helper/model cells, then this
cell; preparation and training do not need to be rerun.


In [ ]:
if RUN_TEST_EVALUATION:
    checkpoint = torch.load(OUTPUT / "best.pt", map_location="cpu", weights_only=True)
    if checkpoint.get("architecture") != "residual-gcn-mlp-v2":
        raise ValueError("Checkpoint is not from this residual GCN/MLP architecture")
    saved = checkpoint["config"]
    if saved["oracle"] != ORACLE_SETTINGS or saved["model"]["input_dim"] != INPUT_DIM:
        raise ValueError("Oracle/feature configuration differs from saved checkpoint")
    test_model = SentenceScorer(**saved["model"]).to(DEVICE)
    test_model.load_state_dict(checkpoint["state_dict"])
    test_ids = canonical_ids("test")
    saved_manifest = json.loads((OUTPUT / "split_manifest.json").read_text(encoding="utf-8"))
    if hashlib.sha256(json.dumps(saved_manifest, sort_keys=True).encode()).hexdigest() != saved["split_manifest_sha256"]:
        raise ValueError("Split manifest no longer matches the saved checkpoint")
    if test_ids != saved_manifest["test"] or set(test_ids) & (set(saved_manifest["train"]) | set(saved_manifest["dev"])):
        raise ValueError("Test manifest changed or overlaps training/validation")
    test = load_split("test", test_ids)
    test_weight = torch.tensor(saved["training"]["effective_pos_weight"], device=DEVICE)
    test_metrics, predictions = evaluate(
        test_model, test, DEVICE, test_weight, **saved["selection"])
    atomic_json(OUTPUT / "test_metrics.json",
                {"checkpoint_epoch": checkpoint["epoch"], "aggregation": "macro ROUGE F1",
                 "reference": "clean_summary", "metrics": test_metrics})
    temporary = OUTPUT / "test_predictions.jsonl.tmp"
    try:
        with temporary.open("w", encoding="utf-8") as handle:
            for row in predictions:
                handle.write(json.dumps(row, ensure_ascii=False) + "\n")
        os.replace(temporary, OUTPUT / "test_predictions.jsonl")
    finally:
        temporary.unlink(missing_ok=True)
    print("Test metrics:", json.dumps(test_metrics), flush=True)
else:
    print("Test not evaluated. Enable RUN_TEST_EVALUATION only after model choices are fixed.")
